# 채소 이미지 분류 학습 결과

학습은 완료되어 model.pth에 저장되어 있습니다. 총 600장 중 480장으로 학습하고 120장으로 검증했습니다. 정확한 출처는 DATA_USED.md를 읽어주세요. 노트북을 이 파일이 든 폴더에서 실행합니다. 필요한 패키지는 README의 설치 명령으로 설치합니다. 그래프를 보려면 matplotlib도 필요합니다.


In [ ]:
from pathlib import Path
import json, csv
import torch
from model_utils import load_model, read_image, CLASS_NAMES
ROOT = Path.cwd()
assert (ROOT / "model.pth").is_file(), "model.pth가 있는 폴더에서 실행하세요."
model, checkpoint = load_model(ROOT / "model.pth", torch.device("cpu"))
print("학습 완료 epoch:", checkpoint["epochs_completed"])
print("선택된 epoch:", checkpoint["best_epoch"])
print(json.dumps(checkpoint["counts"], indent=2, ensure_ascii=False))


## 학습이 일어났는지 확인
분류층의 가중치는 바뀌고, 이번에 고정한 합성곱 본체는 그대로 유지되어야 합니다.


In [ ]:
verification = checkpoint["training_verification"]
print("분류층 변경:", verification["classifier_changed"])
print("고정 본체 유지:", verification["frozen_backbone_unchanged"])
print("학습 파라미터 수:", verification["trainable_parameter_count"])


## 학습 곡선과 검증 결과
이 검증 세트는 모델 선택에 사용했으므로 최종 시험 성능이 아닙니다.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(2, 1, figsize=(12, 12))
for ax, name in zip(axes, ["learning_curves.png", "confusion_matrix.png"]):
    ax.imshow(Image.open(ROOT / name))
    ax.axis("off")
plt.show()
print(json.dumps(checkpoint["metrics"], indent=2, ensure_ascii=False))


## 직접 찍은 사진 예측
my_photos 폴더를 만들고 사진을 넣은 다음 PHOTO를 바꿉니다. 이 사진은 학습에 자동 추가되지 않습니다. 점수는 보정되지 않은 모델 출력입니다.


In [ ]:
PHOTO = ROOT / "my_photos" / "example.jpg"
if PHOTO.is_file():
    with torch.inference_mode():
        probabilities = model(read_image(PHOTO).unsqueeze(0)).softmax(1)[0]
    index = int(probabilities.argmax())
    print("예측:", CLASS_NAMES[index])
    print("모델 점수:", float(probabilities[index]))
    plt.imshow(Image.open(PHOTO)); plt.axis("off"); plt.show()
else:
    print("my_photos에 사진을 넣고 PHOTO 경로를 수정하세요.")


## 재학습 명령
터미널에서 `python train.py --manifest manifest.csv --out retrained --device cpu`를 실행합니다. 이번 모델은 분류층만 학습했습니다. 이상 부위 표시나 별도 손상 유형 분류는 구현되지 않았습니다.
